# Notebook where to build the Tensorflow Recommender Model

## Data Prep

In [1]:
import pandas as pd

customers_csv_path = '/data/processed/customers_df.csv'
customers_df = pd.read_csv(customers_csv_path)
customers_df.head(3)

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
customers_df.shape

(1371980, 8)

In [ ]:
customers_df.columns

Index(['customer_id', 'Active', 'club_member_status', 'fashion_news_frequency',
       'age', 'postal_code', 'purchased_articles', 'favorite_sales_channel'],
      dtype='object')

In [ ]:
customers_df['Active'].unique()

array([0., 1.])

In [ ]:
customers_df['club_member_status'].unique()

array(['ACTIVE', 'LEFT CLUB', 'PRE-CREATE'], dtype=object)

In [ ]:
import numpy as np

np.random.seed(42)

sample_size = 5000
unique_ids = customers_df['customer_id'].unique()
sampled_ids = np.random.choice(unique_ids, size=sample_size, replace=False)
customers_data = customers_df[customers_df['customer_id'].isin(sampled_ids)].copy()

print(f"Sampled Dataset Shape: {customers_data.shape}")

In [ ]:
def encode_status(x):
    if x == "PRE-CREATE":
        return 1
    elif x == "ACTIVE":
        return 2
    else:
        return 0

customers_data['club_member_status_encoded'] = customers_data['club_member_status'].apply(lambda x: encode_status(x))

In [ ]:
customers_data['fashion_news_frequency'].unique()

array(['NONE', 'Regularly', 'Monthly'], dtype=object)

In [ ]:
def encode_news(x):
    if x == "Regularly":
        return 1
    elif x == "Regularly":
        return 2
    else:
        return 0

customers_data['fashion_news_frequency_encoded'] = customers_data['fashion_news_frequency'].apply(lambda x: encode_status(x))

In [ ]:
customers_data['favorite_sales_channel'].unique()

array([ 2.,  1., nan])

In [ ]:
customers_data.fillna({'favorite_sales_channel': 0}, inplace=True)

In [ ]:
customers_data['favorite_sales_channel'].unique()

array([2., 1., 0.])

In [ ]:
customers_data.isna().sum()

customer_id                       0
Active                            0
club_member_status                0
fashion_news_frequency            0
age                               0
postal_code                       0
purchased_articles                0
favorite_sales_channel            0
club_member_status_encoded        0
fashion_news_frequency_encoded    0
dtype: int64

In [ ]:
import ast

customers_data['purchased_articles'] = customers_data['purchased_articles'].apply(lambda x: (ast.literal_eval(x)))
customers_data['nb_past_purchases'] = customers_data['purchased_articles'].apply(lambda x: len(x))
customers_data['nb_past_purchases'].mean().round(0)

np.float64(23.0)

In [ ]:
from itertools import chain

all_articles = set(chain.from_iterable(customers_data["purchased_articles"]))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()
scaler.fit(customers_data[["age"]])
customers_data["age_scaled"] = scaler.transform(customers_data[["age"]]) 

joblib.dump(scaler, "/data/scalers/age_scaler.pkl")

['/data/scalers/age_scaler.pkl']

In [ ]:
customers_data.head(3)

,customer_id,Active,club_member_status,fashion_news_frequency,age,postal_code,purchased_articles,favorite_sales_channel,club_member_status_encoded,fashion_news_frequency_encoded,nb_past_purchases,age_scaled
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49,974 04,"[625548001, 176209023, 627759010, 697138006, 5...",2.0,2,0,21,0.397590
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,ACTIVE,NONE,25,WF1 5EY,"[583558001, 639677008, 640244003, 521269001, 6...",2.0,2,0,86,0.108434
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,ACTIVE,NONE,24,75008,"[663713001, 541518023, 663713001, 578020002, 7...",2.0,2,0,18,0.096386


In [ ]:
import tensorflow as tf
import numpy as np

# 1. Shuffle and split the unique customer IDs
unique_article_ids = np.unique(np.array(list(all_articles)).astype(str))
unique_articles_ds = tf.data.Dataset.from_tensor_slices(unique_article_ids)
unique_customer_ids = np.unique(customers_data['customer_id'].values.astype(str))

shuffled_customers = np.random.permutation(unique_customer_ids)
train_size = int(0.8 * len(shuffled_customers))

train_customers = set(shuffled_customers[:train_size])
test_customers = set(shuffled_customers[train_size:])

# 2. Define our sequences
def sequence_generator(customer_set):
    # Filter the dataframe to only include the relevant customers
    subset_df = customers_data[customers_data['customer_id'].isin(customer_set)]
    
    for _, row in subset_df.iterrows():
        articles = [str(a) for a in row['purchased_articles']]
        if len(articles) < 2:
            continue
            
        for i in range(1, len(articles)):
            start_idx = max(0, i - 10)
            history = articles[start_idx:i]
            padded_history = ["PAD"] * (10 - len(history)) + history
            
            yield (
                {
                    "customer_id": str(row["customer_id"]),
                    "history": padded_history,
                    "age": float(row["age_scaled"]),
                    "active": float(row["Active"]),
                    "status": float(row["club_member_status_encoded"]),
                    "news": float(row["fashion_news_frequency_encoded"]),
                    "channel": float(row["favorite_sales_channel"])
                },
                str(articles[i])
            )

output_signature = (
    {
        "customer_id": tf.TensorSpec(shape=(), dtype=tf.string),
        "history": tf.TensorSpec(shape=(10,), dtype=tf.string),
        "age": tf.TensorSpec(shape=(), dtype=tf.float32),
        "active": tf.TensorSpec(shape=(), dtype=tf.float32),
        "status": tf.TensorSpec(shape=(), dtype=tf.float32),
        "news": tf.TensorSpec(shape=(), dtype=tf.float32),
        "channel": tf.TensorSpec(shape=(), dtype=tf.float32),
    },
    tf.TensorSpec(shape=(), dtype=tf.string)
)

# 3. Create the two TF Datasets
train_ds = tf.data.Dataset.from_generator(lambda: sequence_generator(train_customers), output_signature=output_signature).batch(2048).cache().prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(lambda: sequence_generator(test_customers), output_signature=output_signature).batch(512).cache().prefetch(tf.data.AUTOTUNE)

## Modeling

### Create Model

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
import tensorflow as tf
import tensorflow_recommenders as tfrs


# --- 1. Define the Query Tower (User + History + Metadata) ---
class QueryTower(tf.keras.Model):
    def __init__(self, embedding_dimension, unique_customer_ids, unique_article_ids):
        super().__init__()
        
        # ID Embedding
        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_customer_ids, mask_token=None),
            tf.keras.layers.Embedding(len(unique_customer_ids) + 1, embedding_dimension)
        ])

        # Sequential History with LSTM
        self.history_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_article_ids, mask_token="PAD"),
            # mask_zero=True is critical for LSTMs to ignore PAD tokens
            tf.keras.layers.Embedding(len(unique_article_ids) + 1, embedding_dimension, mask_zero=True),
            tf.keras.layers.LSTM(embedding_dimension)
        ])

        # Feature Fusion MLP
        self.dense_fuser = tf.keras.Sequential([
            tf.keras.layers.Dense(embedding_dimension, activation="relu"),
            tf.keras.layers.Dense(embedding_dimension) 
        ])

    def call(self, inputs):
        # Generate embeddings
        user_emb = self.user_embedding(inputs["customer_id"])
        hist_emb = self.history_embedding(inputs["history"])
        
        # Concatenate metadata
        metadata = tf.concat([
            tf.expand_dims(inputs["age"], -1),
            tf.expand_dims(inputs["status"], -1),
            tf.expand_dims(inputs["active"], -1),
            tf.expand_dims(inputs["news"], -1),
            tf.expand_dims(inputs["channel"], -1)
        ], axis=1)
        
        # Combine all: User ID + LSTM Output + Metadata
        combined = tf.concat([user_emb, hist_emb, metadata], axis=1)
        return self.dense_fuser(combined)

# --- 2. Define the Candidate Tower ---
embedding_dimension = 32

article_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(vocabulary=unique_article_ids, mask_token=None),
    tf.keras.layers.Embedding(len(unique_article_ids) + 1, embedding_dimension)
])

# --- 3. Define the Full Recommender Model ---
class FashionModel(tfrs.Model):
    def __init__(self, query_model, candidate_model, unique_articles_ds):
        super().__init__()
        self.query_model = query_model
        self.candidate_model = candidate_model
        
        # Task & Metrics
        # We pass the pre-built Dataset here to avoid the AttributeError
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=unique_articles_ds.batch(128).map(self.candidate_model)
            )
        )

    def compute_loss(self, inputs, training=False):
        features, labels = inputs

        query_embeddings = self.query_model(features)
        positive_candidate_embeddings = self.candidate_model(labels)
        
        return self.task(query_embeddings, positive_candidate_embeddings)


# --- 4. Initialization and Compilation ---
query_tower = QueryTower(embedding_dimension, unique_customer_ids, unique_article_ids)

model = FashionModel(query_tower, article_model, unique_articles_ds)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01))


### Train

In [ ]:
import datetime

BATCH_SIZE = 2048
EPOCHS = 10

train = train_ds.shuffle(100_000).prefetch(tf.data.AUTOTUNE)

log_dir = os.path.join("/data/models/logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
checkpoint_path = "/data/models/checkpoints/fashion_model_best.weights.h5"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=2, 
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    )
]


NameError: name 'train_ds' is not defined

In [ ]:
model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
results = model.evaluate(test_ds, return_dict=True)
print(f"Test Top-100 Accuracy: {results['factorized_top_k/top_100_categorical_accuracy']:.4f}")

In [ ]:
print(history.history.keys())

In [ ]:
import matplotlib.pyplot as plt

def plot_training_results(history):
    # Set up the figure
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # 1. Plot Training & Validation Loss
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='tab:red')
    ax1.plot(history.history['loss'], label='Train Loss', color='tab:red', linestyle='--')
    if 'val_loss' in history.history:
        ax1.plot(history.history['val_loss'], label='Val Loss', color='red')
    ax1.tick_params(axis='y', labelcolor='tab:red')
    ax1.grid(True, linestyle=':', alpha=0.6)

    # 2. Create a second y-axis for Top-K Accuracy
    ax2 = ax1.twinx()
    ax2.set_ylabel('Accuracy', color='tab:blue')
    
    # Plot Top-100 Accuracy (TFRS default metric name)
    metric_name = 'factorized_top_k/top_100_categorical_accuracy'
    if metric_name in history.history:
        ax2.plot(history.history[metric_name], label='Train Top-100 Acc', color='tab:blue', linestyle='--')
    
    if f'val_{metric_name}' in history.history:
        ax2.plot(history.history[f'val_{metric_name}'], label='Val Top-100 Acc', color='blue')
        
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    # Add legends
    fig.tight_layout()
    fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
    plt.title('Model Training: Loss vs Top-100 Accuracy')
    plt.show()


In [ ]:
plot_training_results(history)